In [ ]:
from jointfm_client import bootstrap_notebook
bootstrap_notebook(add_src_root=True)

# Forecast-Driven Long / Flat / Short Trading on Bitcoin

Worked example of a trading decision rule built on top of `forecast_samples` for a JointFM model trained on **absolute prices**. We download daily Yahoo Finance closes for Bitcoin, a few other major crypto assets, and the most important macro drivers, send the joint history to a JointFM deployment, and apply a long / flat / short classifier on the resulting Bitcoin samples.

## Inputs

- **Classifier target**: `BTC-USD` (the only requested forecast column; the long / flat / short rule acts on its samples).
- **Other crypto majors**: `ETH-USD`, `SOL-USD`, `BNB-USD`, `XRP-USD`. Sent as additional targets so the joint model conditions BTC on the rest of the crypto complex.
- **Macro drivers**: S&P 500 (`^GSPC`), Nasdaq Composite (`^IXIC`), VIX (`^VIX`), 10-year US Treasury yield (`^TNX`), US Dollar index futures (`DX=F`), Gold futures (`GC=F`), WTI crude futures (`CL=F`). These cover risk-on / off equities, real rates, the dollar, and the commodity backdrop — the dominant non-crypto inputs that move BTC.

The Yahoo download requires the optional `notebooks` extra (which also pulls in `pandas`):

```bash
uv add "jointfm-client[notebooks]"
```

## Workflow

1. Discover the deployment's data-generation envelope through `client.health()` (`n_input`, `n_output`, capacity caps).
2. Pull daily closes for every requested symbol via `yfinance`, snap them to a business-day calendar, forward-fill within each series to bridge per-symbol gaps, and keep the most recent `n_input` rows.
3. Forecast all symbols with `forecast_samples` while requesting samples only for `BTC-USD`.
4. Convert each price sample to a holding-period **simple return** relative to the last observed BTC close: `R_h = p_h / p_last - 1`. Simple (not log) returns are used because the Kelly bet multiplier is `(1 + f * R)`.
5. From the empirical return distribution per horizon, derive:
   - **Action**: `long` / `flat` / `short`, gated by tail probabilities crossing configurable dead-zone thresholds. The dead zone should be at least the round-trip transaction cost; shorting can use a wider threshold to absorb borrow costs.
   - **Position size**: the Kelly fraction `f* = argmax_f E[log(1 + f R)]` solved numerically on the empirical samples and capped by a configurable leverage limit.
6. Two decision flavors:
   - **Single horizon** — e.g. 7 business days ahead.
   - **Multi-horizon** — e.g. 5 / 7 / 9 business days ahead. Trade only when every horizon agrees on sign; size with the mean of per-horizon Kelly fractions.

## Interpreting the `position` output

`position` is a **unitless fraction of bankroll** in `[-KELLY_CAP, +KELLY_CAP]`. It comes directly from Kelly's objective `E[log(1 + f * R)]`, where `f` is the fraction of current equity committed to the trade:

- `+1.0` — go long with 100% of equity.
- `-1.0` — go short with 100% of equity.
- `+0.25` — go long with 25% of equity.
- `0.0` — flat (either the gating rule said `flat`, or Kelly disagreed in sign with the gated action).

To turn `position` into something a broker can accept:

```
notional = position * equity                    # USD allocated to the trade (sign = side)
units    = notional / current_price             # coins (or contracts for BTC futures)
```

Two practical knobs:

- **`KELLY_CAP` is the "fractional Kelly" lever.** Full Kelly (`KELLY_CAP = 1.0`) is famously aggressive and assumes the empirical sample distribution equals the true distribution. Most practitioners run quarter- or half-Kelly to absorb model error and reduce drawdowns; lower `KELLY_CAP` to `0.25` or `0.5` to do that without changing the code.
- **Leverage > 1 is possible if you allow it.** Setting `KELLY_CAP = 2.0` lets the rule recommend up to `2.0 * equity` exposure. The feasibility bracket inside `kelly_fraction` keeps `1 + f * R > 0` for every drawn sample, so the suggested size can never bankrupt you on the *sampled* tail — but that guarantee is only as good as the forecast's tail.

This is a client-side rule on top of a probabilistic forecaster. The forthcoming architecture-world model supports native classification/regression heads; this notebook will become redundant once that lands.



In [ ]:
import warnings

import numpy as np
import pandas as pd
import yfinance as yf

from jointfm_client import JointFMClient, plan_forecast_columns

# BTC-USD is the classifier target. Other crypto majors and macro drivers are
# sent alongside as additional targets so the joint model conditions the
# Bitcoin marginal on the rest of the crypto complex and the macro backdrop.
PRICE_TARGET = 'BTC-USD'
ADDITIONAL_CRYPTO_SYMBOLS = ('ETH-USD', 'SOL-USD', 'BNB-USD', 'XRP-USD')
ECONOMIC_DRIVER_SYMBOLS = (
    '^GSPC',     # S&P 500 index level
    '^IXIC',     # Nasdaq Composite index level
    '^VIX',      # CBOE volatility index
    '^TNX',      # 10-year US Treasury yield (quoted as yield-in-percent x 10)
    'DX-Y.NYB',  # ICE US Dollar index spot
    'GC=F',      # Gold futures (USD/oz)
    'CL=F',      # WTI crude oil futures (USD/bbl)
)
ALL_SYMBOLS = (PRICE_TARGET,) + ADDITIONAL_CRYPTO_SYMBOLS + ECONOMIC_DRIVER_SYMBOLS

# Holding horizons (business days ahead of the last observed close).
SINGLE_HORIZON = 7
MULTI_HORIZONS = (5, 7, 9)
HORIZONS = tuple(sorted({SINGLE_HORIZON, *MULTI_HORIZONS}))

# Monte-Carlo sample budget for the empirical return distribution per horizon.
N_SAMPLES = 10_000
SEED = 7

# Dead-zone thresholds, in simple-return units per holding period.
# 50 bps ~ round-trip cost; shorting carries borrow cost on top.
EPS_LONG = 0.005
EPS_SHORT = 0.010

# Minimum tail probability above the threshold required to take a position.
CONFIDENCE_THRESHOLD = 0.55

# Hard cap on |position|. 1.0 = fully invested, <1.0 = de-levered.
KELLY_CAP = 1.0



In [ ]:
client = JointFMClient.from_env()
health = client.health(cache=True)
capacity = health.data_generation
if capacity is None:
    raise ValueError('Deployment /healthz response is missing the data_generation block')
if max(HORIZONS) > capacity.n_output:
    raise ValueError(
        f'Largest horizon {max(HORIZONS)} exceeds deployment n_output={capacity.n_output}'
    )

# yfinance pins a few internal calls to the soon-to-be-removed pandas
# Timestamp.utcnow API; mute that upstream-only deprecation noise here so
# the notebook output stays readable. Errors and our own warnings still bubble.
with warnings.catch_warnings():
    warnings.filterwarnings(
        'ignore',
        message=r'Timestamp\.utcnow is deprecated',
        module=r'yfinance\..*',
    )
    raw_yahoo = yf.download(
        list(ALL_SYMBOLS),
        period='max',
        interval='1d',
        auto_adjust=False,
        progress=False,
        actions=False,
        threads=False,
    )
if not isinstance(raw_yahoo.columns, pd.MultiIndex) or 'Close' not in raw_yahoo.columns.get_level_values(0):
    raise ValueError('Yahoo Finance response does not expose a Close field')
close_frame = raw_yahoo['Close'].sort_index()
missing_symbols = [symbol for symbol in ALL_SYMBOLS if symbol not in close_frame.columns]
if missing_symbols:
    raise ValueError(f'Yahoo Finance did not return Close columns for {missing_symbols!r}')
close_frame = close_frame.loc[:, list(ALL_SYMBOLS)].astype(float)

# Crypto trades 7 days a week, equities and futures 5 days. Snap to a
# business-day calendar and forward-fill within each series so per-symbol
# holidays do not punch holes through the joint history.
business_index = pd.date_range(close_frame.index.min(), close_frame.index.max(), freq='B')
business_close = close_frame.reindex(business_index).ffill()

# A column is "dead" when Yahoo returned the symbol but every row is NaN
# (typical for delisted/renamed tickers like the retired DX=F future).
# Flag those explicitly instead of letting first_valid_index() return None
# and trip max() with an unrelated TypeError.
first_valid_per_symbol = {
    symbol: business_close[symbol].first_valid_index() for symbol in ALL_SYMBOLS
}
dead_symbols = [symbol for symbol, ts in first_valid_per_symbol.items() if ts is None]
if dead_symbols:
    raise ValueError(
        f'Yahoo Finance returned no usable Close data for {dead_symbols!r}; '
        f'the ticker may have been delisted or renamed. Replace it in '
        f'ECONOMIC_DRIVER_SYMBOLS / ADDITIONAL_CRYPTO_SYMBOLS.'
    )

# Trim leading rows where any symbol has not yet started trading (e.g. SOL-USD
# lists only from 2020). n_input is the deployment's training-time upper bound,
# so feed at most that many of the most recent business days; shorter joint
# histories are accepted by the service.
common_start = max(first_valid_per_symbol.values())
aligned = business_close.loc[common_start:]
if aligned.isna().any().any():
    bad_columns = aligned.columns[aligned.isna().any()].tolist()
    raise ValueError(f'Residual NaN after alignment in columns: {bad_columns!r}')
if len(aligned) == 0:
    raise ValueError(f'No aligned Yahoo history available for {list(ALL_SYMBOLS)!r}')
history_length = min(capacity.n_input, len(aligned))
history = aligned.iloc[-history_length:].copy().reset_index(drop=True)

last_btc_price = float(history[PRICE_TARGET].iloc[-1])
if last_btc_price <= 0.0:
    raise ValueError(f'Last observed {PRICE_TARGET} must be positive, got {last_btc_price}')

window_start = aligned.index[-history_length].date()
window_end = aligned.index[-1].date()
print(
    f'History window: {window_start} -> {window_end} '
    f'({history_length} business days; deployment cap n_input={capacity.n_input})'
)
print(f'Symbols ({len(ALL_SYMBOLS)}): {list(ALL_SYMBOLS)}')
print(f'Last observed {PRICE_TARGET}: {last_btc_price:,.2f}')
print(f'Forecast horizons (business days ahead): {HORIZONS}')



In [ ]:
# query_time for h-step-ahead is len(history) - 1 + h (matches the convention
# in forecast_samples.ipynb where the last history row corresponds to h=1).
query_times = [len(history) - 1 + h for h in HORIZONS]

# All Yahoo symbols are passed as targets so the plan stays valid on
# max_features=0 deployments. plan_forecast_columns downgrades features to
# targets automatically when the deployment allows it.
plan = plan_forecast_columns(
    health=health,
    feature_columns=(),
    target_columns=list(ALL_SYMBOLS),
    history_length=len(history),
    query_times_length=len(query_times),
)
result = client.forecast_samples(
    history,
    query_times=query_times,
    requested_columns=[PRICE_TARGET],
    columns=plan.columns,
    n_samples=N_SAMPLES,
    seed=SEED,
)

price_samples = result.to_numpy()  # (sample, horizon, column)
expected_shape = (N_SAMPLES, len(HORIZONS), 1)
if price_samples.shape != expected_shape:
    raise ValueError(f'Expected sample tensor of shape {expected_shape}, got {price_samples.shape}')
price_samples = price_samples[:, :, 0]  # drop the singleton requested-column axis



In [ ]:
# BTC-USD cannot trade below zero in the real world. The joint model emits an
# unbounded continuous marginal, so its tail can occasionally cross zero on
# absolute-price targets. Clamp such samples at 0 (capping the worst-case
# holding-period return at -100%) and surface a warning so a fat negative tail
# does not silently degrade calibration.
negative_count = int(np.sum(price_samples < 0.0))
if negative_count > 0:
    fraction = negative_count / price_samples.size
    print(
        f'WARNING: {negative_count} of {price_samples.size} {PRICE_TARGET} samples '
        f'({fraction:.2%}) were negative; clamped to 0.0.'
    )
    price_samples = np.maximum(price_samples, 0.0)

simple_returns_by_horizon = {
    horizon: price_samples[:, index] / last_btc_price - 1.0
    for index, horizon in enumerate(HORIZONS)
}

pd.DataFrame(
    {f'R_{horizon}': pd.Series(simple_returns_by_horizon[horizon]).describe() for horizon in HORIZONS}
).round(6)



In [ ]:
KELLY_BISECTION_ITERATIONS = 80
FEASIBLE_BRACKET_SAFETY = 1e-6


def decision_probabilities(returns: np.ndarray, *, eps_long: float, eps_short: float) -> dict[str, float]:
    """Tail probabilities of crossing the long / short thresholds.

    Returns the empirical mass strictly above ``+eps_long``, strictly below ``-eps_short``,
    and the residual mass inside the dead zone.
    """
    p_long = float(np.mean(returns > eps_long))
    p_short = float(np.mean(returns < -eps_short))
    return {'p_long': p_long, 'p_short': p_short, 'p_flat': 1.0 - p_long - p_short}


def kelly_fraction(returns: np.ndarray, *, max_leverage: float) -> float:
    """Numerical Kelly fraction maximizing ``E[log(1 + f R)]`` on empirical samples.

    Returns the unconstrained interior optimum when one exists inside the feasible
    bracket ``(-1/R_max, -1/R_min)`` (where ``1 + f R > 0`` for every sample),
    clipped to ``[-max_leverage, +max_leverage]``. Returns the closest boundary
    when the derivative does not change sign on the bracket.
    """
    r = np.asarray(returns, dtype=float)
    r_max = float(r.max())
    r_min = float(r.min())
    feasible_upper = (-1.0 / r_min) if r_min < 0.0 else float('inf')
    feasible_lower = (-1.0 / r_max) if r_max > 0.0 else float('-inf')
    f_upper = min(feasible_upper - FEASIBLE_BRACKET_SAFETY, max_leverage)
    f_lower = max(feasible_lower + FEASIBLE_BRACKET_SAFETY, -max_leverage)
    if f_lower >= f_upper:
        return 0.0

    def derivative(f: float) -> float:
        """First-order condition of ``E[log(1 + f R)]`` evaluated at ``f``."""
        return float(np.mean(r / (1.0 + f * r)))

    g_lower = derivative(f_lower)
    g_upper = derivative(f_upper)
    if g_lower > 0.0 and g_upper > 0.0:
        return f_upper
    if g_lower < 0.0 and g_upper < 0.0:
        return f_lower
    a, b = f_lower, f_upper
    for _ in range(KELLY_BISECTION_ITERATIONS):
        m = 0.5 * (a + b)
        if derivative(m) > 0.0:
            a = m
        else:
            b = m
    return 0.5 * (a + b)


def decide(
    returns: np.ndarray,
    *,
    eps_long: float,
    eps_short: float,
    confidence_threshold: float,
    kelly_cap: float,
) -> dict[str, float | str]:
    """Combine tail-probability gating with a Kelly-sized position.

    The action is ``long`` or ``short`` only when the corresponding tail probability
    clears ``confidence_threshold`` and dominates the other tail; otherwise the
    action is ``flat`` and the position is zero. When the Kelly optimum disagrees
    in sign with the gated action, the position is clipped to zero.
    """
    probabilities = decision_probabilities(returns, eps_long=eps_long, eps_short=eps_short)
    p_long = probabilities['p_long']
    p_short = probabilities['p_short']
    if p_long >= confidence_threshold and p_long > p_short:
        action = 'long'
    elif p_short >= confidence_threshold and p_short > p_long:
        action = 'short'
    else:
        action = 'flat'
    raw_kelly = kelly_fraction(returns, max_leverage=kelly_cap)
    if action == 'long':
        position = float(np.clip(raw_kelly, 0.0, kelly_cap))
    elif action == 'short':
        position = float(np.clip(raw_kelly, -kelly_cap, 0.0))
    else:
        position = 0.0
    return {
        'action': action,
        'p_long': p_long,
        'p_short': p_short,
        'p_flat': probabilities['p_flat'],
        'mean_return': float(np.mean(returns)),
        'std_return': float(np.std(returns, ddof=1)),
        'kelly_fraction': raw_kelly,
        'position': position,
    }


## Single horizon

Decide at one fixed holding period (`SINGLE_HORIZON` = 7 steps ahead). The reported row contains the action, the long/short/flat tail probabilities, the empirical mean/std of the simple return, the unconstrained Kelly optimum, and the gated, capped position.


In [ ]:
single_decision = decide(
    simple_returns_by_horizon[SINGLE_HORIZON],
    eps_long=EPS_LONG,
    eps_short=EPS_SHORT,
    confidence_threshold=CONFIDENCE_THRESHOLD,
    kelly_cap=KELLY_CAP,
)
pd.Series(single_decision, name=f'horizon_{SINGLE_HORIZON}').to_frame()


## Multi-horizon

Compute the same decision independently for each horizon in `MULTI_HORIZONS` = (5, 7, 9). The combined rule trades only when every horizon agrees on sign (`long` everywhere or `short` everywhere) and sizes the position with the mean of per-horizon Kelly fractions, capped by `KELLY_CAP`. Disagreement collapses to `flat` — the cheapest way to encode "I am not confident across the holding-period choice".


In [ ]:
per_horizon = pd.DataFrame(
    {
        f'horizon_{horizon}': decide(
            simple_returns_by_horizon[horizon],
            eps_long=EPS_LONG,
            eps_short=EPS_SHORT,
            confidence_threshold=CONFIDENCE_THRESHOLD,
            kelly_cap=KELLY_CAP,
        )
        for horizon in MULTI_HORIZONS
    }
).T

action_set = set(per_horizon['action'].tolist())
if action_set == {'long'}:
    combined_action = 'long'
elif action_set == {'short'}:
    combined_action = 'short'
else:
    combined_action = 'flat'

mean_kelly = float(per_horizon['kelly_fraction'].mean())
if combined_action == 'long':
    combined_position = float(np.clip(mean_kelly, 0.0, KELLY_CAP))
elif combined_action == 'short':
    combined_position = float(np.clip(mean_kelly, -KELLY_CAP, 0.0))
else:
    combined_position = 0.0

print(f'Combined multi-horizon action  : {combined_action}')
print(f'Combined Kelly-sized position  : {combined_position:+.4f}')
per_horizon
